In [2]:
import numpy as np
import pandas as pd
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectPercentile
from sklearn.model_selection import RandomizedSearchCV, RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import joblib

In [4]:
DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"

TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, "train-dnam")
X_train_path = os.path.join(TRAIN_DATASET_PATH, "features.dnam.csv")
y_train_path = os.path.join(TRAIN_DATASET_PATH, "labels.csv")

output_dir = "/home/mariya/Data/pallas/models/dnam"
random_seed = 42
n_jobs = 10
selected_features_path = os.path.join(output_dir, "selected_features.txt")
pipeline_path = os.path.join(output_dir, "pipeline.pkl")
hyperparams_path = os.path.join(output_dir, "hyperparams.pkl")

# Print configuration
print(f"X_train_path: {X_train_path}")
print(f"y_train_path: {y_train_path}")
print(f"output_dir: {output_dir}")
print(f"random_seed: {random_seed}")
print(f"n_jobs: {n_jobs}")
print(f"selected_features_path: {selected_features_path}")
print(f"pipeline_path: {pipeline_path}")
print(f"hyperparams_path: {hyperparams_path}")

# Hardcoded variables
FEATURE_SELECTION_PERCENTILE=10  # KEEP TOP n% OF CPG SITES
FINAL_N_ESTIMATORS=500

X_train_path: /home/mariya/Data/pallas/mlomix_datasets/train-dnam/features.dnam.csv
y_train_path: /home/mariya/Data/pallas/mlomix_datasets/train-dnam/labels.csv
output_dir: /home/mariya/Data/pallas/models/dnam
random_seed: 42
n_jobs: 10
selected_features_path: /home/mariya/Data/pallas/models/dnam/selected_features.txt
pipeline_path: /home/mariya/Data/pallas/models/dnam/pipeline.pkl
hyperparams_path: /home/mariya/Data/pallas/models/dnam/hyperparams.pkl


In [ ]:
# Read in X_train and y_train
X_train = pd.read_csv(X_train_path, index_col=0)
X_train


In [ ]:
y_train = pd.read_csv(y_train_path, index_col=0)
y_train

,class
id,
ALL_1,"B-ALL, high hyperdiploid"
ALL_1002,"B-ALL, high hyperdiploid"
ALL_1004,"B-ALL, PAX5alt"
ALL_1007,"B-ALL, high hyperdiploid"
ALL_1008,"B-ALL, high hyperdiploid"
...,...
TCGA-AB-2997,"AML, APL-PML::RARA"
TCGA-AB-2999,"AML, APL-PML::RARA"
TCGA-AB-3001,"AML, APL-PML::RARA"


In [ ]:
# Show class distributions in y_train and y_test
print("y_train class distribution:")
print(y_train.value_counts())


y_train class distribution:
class                   
AML, KMT2A-r                246
B-ALL, high hyperdiploid    229
B-ALL, ETV6::RUNX1          199
AML, RUNX1::RUNX1T1         173
AML, NPM1alt                131
AML, CBFB::MYH11            126
Non-leukemic                 95
T-ALL                        73
AML, NUP98-r                 68
AML, CEBPAalt                55
B-ALL, KMT2A-r               43
B-ALL, PAX5alt               35
B-ALL, TCF3::PBX1            26
AML, CBFA2T3::GLIS2          22
AML, DEK::NUP214             19
B-ALL, BCR::ABL1             17
AML, APL-PML::RARA           16
AML, ETV6-r                  16
B-ALL, iAMP21                15
B-ALL, DUX4-r                14
B-ALL, ZNF384-r              12
B-ALL, ETV6::RUNX1-like      10
B-ALL, BCR::ABL1-like         8
AML, KAT6A-r                  8
AML, RBM15::MKL1              8
B-ALL, hypodiploid            7
B-ALL, PAX5 P80R              6
AML, FUS::ERG                 6
B-ALL, MEF2D-r                6
AML, MECOM-r       

In [ ]:
# Get least frequently ocurring class count
min_class_count = y_train.value_counts().min()
min_class_count

np.int64(5)

In [ ]:
y_train = y_train.iloc[:, 0].values  # Convert to 1D numpy array

In [ ]:
# Reduce memory usage by converting to float32
X_train = X_train.astype(np.float32)

In [19]:
N_ITER=30
N_REPEATS=1 # WE WILL VALIDATE FINAL PARAMS USING MORE REPEATS
N_FOLDS_INITIAL=3 # down from min_class_count to speed up initial search

# ----------------------
# Pipeline: RF feature selection -> RF classifier
# ----------------------
pipeline = Pipeline([
    ("feature_selection", SelectPercentile(percentile=FEATURE_SELECTION_PERCENTILE)),  # keep only top 10% of features
    ("rf", RandomForestClassifier(random_seed, n_jobs=-n_jobs)) # inner: RF uses all cores
])

# ----------------------
# Hyperparameter search space
# ----------------------
param_grid = {
    'rf__n_estimators': [50],  # We will increase this in the final model
    'rf__max_depth': [None, 20, 50],
    'rf__max_features': ['sqrt', 'log2', 0.25],
    'rf__min_samples_split': [2, 5, 10],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__class_weight': ['balanced']
}

# ----------------------
# Cross-validation + randomized search
# ----------------------
cv = RepeatedStratifiedKFold(n_splits=N_FOLDS_INITIAL,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_grid,
    n_iter=N_ITER,
    cv=cv,
    scoring="f1_weighted",   # good for multiclass imbalance
    n_jobs=1, # outer loop: sequential
    random_state=random_seed,
    verbose=1
)

search.fit(X_train, y_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV (F1-weighted):", search.best_score_)
best_pipeline = search.best_estimator_

## SAVE EVERYTHING
# Save best hyperparameters
joblib.dump(search.best_params_, hyperparams_path)

# get features selected by SelectFromModel
selected_mask = best_pipeline.named_steps['feature_selection'].get_support()
selected_features = X_train.columns[selected_mask].tolist()

# Write selected features to a text file
with open(selected_features_path, 'w') as f:
    for feature in selected_features:
        f.write(f"{feature}\n")

# Save the whole pipeline
joblib.dump(best_pipeline, pipeline_path)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best hyperparameters: {'rf__n_estimators': 50, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 1, 'rf__max_features': 0.25, 'rf__max_depth': 20, 'rf__class_weight': 'balanced'}
Best CV (F1-weighted): 0.8811226235411319


['/home/mariya/Data/pallas/models/dnam/pipeline.pkl']

In [20]:
# Cross-validate again with more repeats and larger estimators
N_REPEATS=10

# Use just the selected features
X_train_selected = X_train[selected_features]

# Read in "best_hyperparameters.pkl"
with open(hyperparams_path, "rb") as f:
    best_params = joblib.load(f)

best_params = {k.replace('rf__', ''): v for k, v in best_params.items()}
best_params['n_estimators'] = FINAL_N_ESTIMATORS
print("Final model parameters:", best_params)
final_rf = RandomForestClassifier(**best_params,
                            n_jobs=n_jobs,
                            random_state=random_seed)

# Cross-validate again with more repeats
cv = RepeatedStratifiedKFold(n_splits=min_class_count,
                             n_repeats=N_REPEATS,
                             random_state=random_seed)

cv_scores = cross_val_score(final_rf,
                           X_train_selected,
                           y_train,
                           cv=cv,
                           scoring="f1_weighted",
                           n_jobs=1,
                           verbose=1)

print(f"Cross-validation F1-weighted: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Train the final model on all training data
final_rf.fit(X_train_selected, y_train)

# Save the final model
joblib.dump(final_rf, os.path.join(output_dir, "final_model.pkl"))

Final model parameters: {'n_estimators': 500, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.25, 'max_depth': 20, 'class_weight': 'balanced'}


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed: 395.2min
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed: 402.9min finished


Cross-validation F1-weighted: 0.891 ± 0.014


['/home/mariya/Data/pallas/models/dnam/final_model.pkl']